# Financial MCQ Benchmark: Qwen2.5-14B Zero-Shot Evaluation

Single-model, multi-dataset, multilingual **zero-shot** benchmarking on financial MCQ datasets.

**Datasets used:**
- CFA / CPA (English)
- ES-MultiFinQA (Spanish)
- Plutus-MultiFinQA (multilingual)
- Arabic Accounting MCQ (Arabic)
- Arabic Business MCQ (Arabic)

> **Note:** BhashaBench Finance is excluded — it is a gated dataset requiring HuggingFace approval.

**Model evaluated:**
- `Qwen/Qwen2.5-14B-Instruct` (~14B parameters)

**Metric**: Accuracy — proportion of correctly identified answer labels.  
**Answer format**: index `0–3` (0 = A, 1 = B, 2 = C, 3 = D).

## Cell 1 — Install Required Libraries

In [ ]:
!pip install transformers datasets peft bitsandbytes accelerate
!pip install sentencepiece protobuf evaluate scikit-learn tqdm
!pip install sacremoses langdetect
!pip install -U bitsandbytes>=0.46.1 accelerate transformers

## Cell 2 — HuggingFace Login

Qwen2.5-14B is a gated model — you must accept the licence on huggingface.co and authenticate.

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# Load token securely from Kaggle Secrets
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("hf_token")

# Login
login(token=hf_token)

## Cell 3 — Load All Datasets from GitHub

Pre-cleaned task1 JSON files are fetched from `hassan09070/clef_task/task1_data/` on GitHub
(local file used first when running locally).

Internal schema per row:
- `question`: `str`
- `choices`: list of **N strings** — variable length, not capped to 4
- `keys`: list of option letters matching `choices`, e.g. `['a','b','c']`
- `gold`: **list of correct letter(s)**, e.g. `['c']` or `['a','b']` (multi-answer aware)
- `source`: `str` — dataset identifier

In [ ]:
import requests
import json
import base64
import os

# ── GitHub source config ───────────────────────────────────────────────────
_GITHUB_REPO   = "hassan09070/clef_task"
_GITHUB_FOLDER = "task1_data"
_GITHUB_TOKEN  = "YOUR_GITHUB_TOKEN_HERE"


def _get_github_token() -> str:
    try:
        from kaggle_secrets import UserSecretsClient
        t = UserSecretsClient().get_secret("github_token")
        if t:
            return t
    except Exception:
        pass
    return os.environ.get("GITHUB_TOKEN") or _GITHUB_TOKEN


def _github_fetch(filename: str) -> list:
    """
    Fetch a JSON file from the private GitHub repo via the Contents API.
    Handles files of any size:
      - ≤1 MB  → inline base64 content in the API response
      - >1 MB  → API returns download_url; fetched separately with auth header
                 (affects: task1_Tomas08119993_finmmeval-cfa-cpa.json ~1 MB
                            task1_bharatgenai_BhashaBench-Finance-Hindi.json ~5 MB)
    """
    token   = _get_github_token()
    headers = {
        "Accept":        "application/vnd.github+json",
        "Authorization": f"token {token}",
    }
    api_url = (
        f"https://api.github.com/repos/{_GITHUB_REPO}/contents/"
        f"{_GITHUB_FOLDER}/{filename}"
    )
    meta = requests.get(api_url, headers=headers, timeout=60)
    meta.raise_for_status()
    data = meta.json()

    content_b64 = data.get("content", "").replace("\n", "")
    if content_b64:
        # Small file — content is base64-encoded inline
        return json.loads(base64.b64decode(content_b64).decode("utf-8"))

    # Large file (>1 MB) — use the download_url provided by the API
    download_url = data.get("download_url")
    if not download_url:
        raise ValueError(f"No content or download_url for {filename}")
    raw = requests.get(download_url, headers={"Authorization": f"token {token}"}, timeout=120)
    raw.raise_for_status()
    return raw.json()


# ── Dataset file mapping ───────────────────────────────────────────────────
_DATASET_FILES = {
    "cfa_cpa":           "task1_Tomas08119993_finmmeval-cfa-cpa.json",
    "es_multifin":       "task1_TheFinAI_flare-es-multifin.json",
    "plutus":            "task1_TheFinAI_plutus-multifin.json",
    "arabic_accounting": "task1_SahmBenchmark_arabic-accounting-mcq.json",
    "arabic_business":   "task1_SahmBenchmark_arabic-business-mcq.json",
    "hindi_finance":    "task1_bharatgenai_BhashaBench-Finance-Hindi.json"
}


def load_task1_json(filename: str, source_name: str) -> list:
    records    = _github_fetch(filename)
    rows       = []
    for rec in records:
        options = rec.get("options") or {}
        if not options:
            continue
        sorted_keys = sorted(options.keys())
        choices     = [options[k] for k in sorted_keys]
        gold_raw    = rec.get("gold") or []
        valid_gold  = [g.lower() for g in gold_raw if g.lower() in sorted_keys]
        if not valid_gold:
            continue
        rows.append({
            "question": str(rec.get("question") or ""),
            "choices":  choices,
            "keys":     sorted_keys,
            "gold":     valid_gold,
            "source":   source_name,
        })
    return rows


# ── Load all datasets ──────────────────────────────────────────────────────
print(f"Loading from github.com/{_GITHUB_REPO} (private, authenticated)\n")

all_data = []
for src_name, fname in _DATASET_FILES.items():
    print(f"  {src_name:<33}", end="  ", flush=True)
    try:
        rows = load_task1_json(fname, src_name)
        all_data.extend(rows)
        print(f"{len(rows):>5} rows")
    except Exception as e:
        print(f"FAILED — {e}")

print(f"\nTotal loaded: {len(all_data)} examples")

In [ ]:
# ES-MultiFinQA, Plutus, BhashaBench, Arabic Accounting, Arabic Business
# are all loaded in the cell above via GitHub JSON files.
# This cell is intentionally left empty.

In [ ]:
# See unified GitHub loader above.

In [ ]:
# BhashaBench Finance — add entry to _DATASET_FILES above and upload the
# task1_bharatgenai_BhashaBench-Finance-Hindi.json to the GitHub repo to include it.

In [ ]:
# See unified GitHub loader above.

In [ ]:
# See unified GitHub loader above.

## Cell 4 — Merge All Datasets into One Final Dataset

In [ ]:
from collections import Counter

# all_data is the merged list produced by the GitHub loader above.
print(f"Total examples: {len(all_data)}\n")

print("Source breakdown (with choice-count distribution):")
for src, cnt in sorted(Counter(r["source"] for r in all_data).items(), key=lambda x: -x[1]):
    n_choices = Counter(len(r["choices"]) for r in all_data if r["source"] == src)
    choices_str = "  ".join(f"{k}opt:{v}" for k, v in sorted(n_choices.items()))
    print(f"  {src:<35} {cnt:>5}   ({choices_str})")

In [ ]:
from collections import Counter

# Gold-letter distribution (multi-label: each row may have >1 correct answer)
all_gold = [g for row in all_data for g in row["gold"]]
gold_dist = Counter(all_gold)
total_q   = len(all_data)

print("Gold-letter distribution:")
for letter in sorted(gold_dist):
    cnt = gold_dist[letter]
    bar = "█" * int(cnt / total_q * 40)
    print(f"  {letter.upper()}: {bar} {cnt:>4}  ({cnt/total_q*100:.1f}%)")

multi_gold = sum(1 for row in all_data if len(row["gold"]) > 1)
print(f"\nMulti-gold rows (>1 correct answer): {multi_gold}")

# Variable choice-count distribution
choice_counts = Counter(len(row["choices"]) for row in all_data)
print("\nChoice-count distribution:")
for n, cnt in sorted(choice_counts.items()):
    print(f"  {n} choices: {cnt:>5} questions")

# Sample walkthrough
sample = all_data[0]
print(f"\nSample from all_data[0]  (source: {sample['source']}):")
print(f"  question : {sample['question'][:100]}…")
for k, ch in zip(sample["keys"], sample["choices"]):
    marker = "  ◄ correct" if k in sample["gold"] else ""
    print(f"  {k.upper()}. {ch[:70]}{marker}")
print(f"  gold     : {sample['gold']}")

## Cell 5 — Benchmark Utility Functions

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import gc


def load_model_4bit(model_name: str):
    """Load a causal LM in 4-bit NF4 quantisation to fit on consumer GPUs."""
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    model.eval()
    print(f"Loaded: {model_name}")
    return model, tokenizer


def free_model(model):
    """Release GPU memory after benchmarking a model."""
    del model
    gc.collect()
    torch.cuda.empty_cache()


def build_prompt(tokenizer, question: str, choices: list) -> str:
    """
    Build a chat-formatted MCQ prompt for variable-length choice lists.
    Labels: A, B, C, D, E, … (auto-extended beyond 4 if needed).
    """
    n      = len(choices)
    labels = [chr(ord("A") + i) for i in range(n)]
    options_str   = "\n".join(f"{labels[i]}. {choices[i]}" for i in range(n))
    valid_letters = "/".join(labels)
    user_msg = (
        "Answer the following multiple-choice question. "
        f"Respond with ONLY the letter of the correct answer ({valid_letters}).\n\n"
        f"Question: {question}\n\n{options_str}"
    )
    if getattr(tokenizer, "chat_template", None):
        messages = [{"role": "user", "content": user_msg}]
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    return f"[INST] {user_msg} [/INST]"


def predict_answer(model, tokenizer, question: str, choices: list, keys: list) -> str:
    """
    Logit-based MCQ prediction for variable-length choice lists.

    Scores each available option by the log-probability of its display label
    (A, B, C, …) at the last input position, then returns the corresponding
    lowercase key letter (e.g. 'a', 'b', 'c') so it can be checked directly
    against the gold list.

    Parameters
    ----------
    choices : list[str]   — option texts, variable length
    keys    : list[str]   — lowercase letter keys matching choices,
                            e.g. ['a','b','c'] — used to map prediction back to gold
    """
    n              = len(choices)
    display_labels = [chr(ord("A") + i) for i in range(n)]   # ['A','B','C',…]

    # Bare letter token ids — avoids BOS / space-prefix ambiguity on some models
    label_token_ids = {
        lbl: tokenizer.convert_tokens_to_ids(lbl)
        for lbl in display_labels
    }

    prompt = build_prompt(tokenizer, question, choices)
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=2048
    ).to(model.device)

    with torch.no_grad():
        # use_cache=False avoids DynamicCache.from_legacy_cache error in transformers>=4.45
        logits = model(**inputs, use_cache=False).logits[0, -1, :]

    best_display = max(display_labels, key=lambda lbl: logits[label_token_ids[lbl]].item())
    best_idx     = display_labels.index(best_display)
    return keys[best_idx]    # lowercase key, e.g. 'a'


def run_benchmark(model, tokenizer, data: list, model_name: str) -> tuple:
    """
    Run individual zero-shot MCQ inference over every row in `data`.

    Each prediction is evaluated one row at a time (no batching).
    A prediction is marked correct when the returned letter is present in
    the row's gold list — handles both single-answer and multi-answer MCQs.

    Parameters
    ----------
    data : list[dict]  — rows with keys: question, choices, keys, gold, source

    Returns
    -------
    (accuracy_float, predictions_list_of_str)
        predictions is a list of lowercase letter strings, e.g. ['a','c','b',…]
    """
    correct     = 0
    predictions = []
    n           = len(data)

    print(f"\n{'='*60}")
    print(f"Benchmarking : {model_name}")
    print(f"Questions    : {n}")
    print(f"{'='*60}")

    for i, row in enumerate(data):
        pred = predict_answer(
            model, tokenizer,
            row["question"],
            row["choices"],
            row["keys"],
        )
        predictions.append(pred)
        if pred in row["gold"]:     # multi-gold aware
            correct += 1
        if (i + 1) % 200 == 0 or (i + 1) == n:
            print(f"  [{i+1:>5}/{n}]  running accuracy: {correct/(i+1)*100:.2f}%")

    accuracy = correct / n * 100
    print(f"\n► {model_name}  —  Final Accuracy: {accuracy:.2f}%  ({correct}/{n})")
    return accuracy, predictions


# Results container — initialised here to avoid accidental state reset
benchmark_results = {}

print("Benchmark utilities ready.")
print("predict_answer  → returns lowercase letter key (e.g. 'a')")
print("run_benchmark   → multi-gold aware, individual inference (no batching)")

## Cell 6 — Model Registry

Single model: **Qwen2.5-14B-Instruct**  
~14B parameters — requires ~20–24 GB VRAM with 4-bit NF4 quantisation (e.g. A100 40 GB).  
Make sure you have accepted the model licence on [huggingface.co/Qwen/Qwen2.5-14B-Instruct](https://huggingface.co/Qwen/Qwen2.5-14B-Instruct).

In [ ]:
MODEL_QWEN14B = "Qwen/Qwen2.5-14B-Instruct"

print("Model to benchmark:")
print(f"  {MODEL_QWEN14B}")
print(f"\nTotal questions in all_data: {len(all_data)}")

## Benchmark Execution

The model is loaded in 4-bit NF4 quantisation.  

**Evaluation method**: For each question the model scores options A/B/C/D by the  
log-probability assigned to that letter token at the last input position.  
The highest-scoring letter is taken as the prediction.

### Qwen2.5-14B-Instruct (`Qwen/Qwen2.5-14B-Instruct`)
~14B parameters — requires ~20–24 GB VRAM with 4-bit quantisation.

In [ ]:
model, tokenizer = load_model_4bit(MODEL_QWEN14B)

acc, preds = run_benchmark(model, tokenizer, all_data, MODEL_QWEN14B)
benchmark_results[MODEL_QWEN14B] = {"accuracy": acc, "predictions": preds}

free_model(model)
del tokenizer

## Results: Accuracy Summary

**Accuracy** = proportion of correctly identified answer labels (0–3) across all questions.  
Random baseline = 25.0 %

In [ ]:
import pandas as pd
from collections import Counter

# ── Overall accuracy table ─────────────────────────────────────────────────
rows = []
for model_name, res in benchmark_results.items():
    preds   = res["predictions"]
    correct = sum(preds[i] in all_data[i]["gold"] for i in range(len(preds)))
    rows.append({
        "Model":        model_name.split("/")[-1],
        "Full Name":    model_name,
        "Accuracy (%)": round(res["accuracy"], 2),
        "Correct":      correct,
        "Total":        len(all_data),
    })

results_df = (
    pd.DataFrame(rows)
    .sort_values("Accuracy (%)", ascending=False)
    .reset_index(drop=True)
)
print("=" * 65)
print("               BENCHMARK RESULTS SUMMARY")
print("=" * 65)
print(results_df[["Model", "Accuracy (%)", "Correct", "Total"]].to_string(index=False))
print(f"\nRandom baseline (variable-choice MCQ): ~25–33 %")
print("Correct = predicted letter is in the gold list (multi-answer aware)")

# ── Per-source breakdown ───────────────────────────────────────────────────
if benchmark_results:
    sources     = sorted(set(row["source"] for row in all_data))
    src_indices = {
        src: [i for i, row in enumerate(all_data) if row["source"] == src]
        for src in sources
    }
    model_short = [m.split("/")[-1][:22] for m in benchmark_results]
    print("\n\nPer-source accuracy breakdown:")
    header = f"  {'Source':<33}" + "".join(f"{m:>24}" for m in model_short)
    print(header)
    print("  " + "-" * (len(header) - 2))
    for src in sources:
        idxs     = src_indices[src]
        row_str  = f"  {src:<33}"
        for model_name, res in benchmark_results.items():
            p = res["predictions"]
            c = sum(p[i] in all_data[i]["gold"] for i in idxs)
            row_str += f"{c/len(idxs)*100:>23.1f}%"
        print(row_str)

## Save Results

In [ ]:
import json

# Save full predictions + accuracies to JSON
output = {}
for model_name, res in benchmark_results.items():
    preds   = res["predictions"]
    correct = sum(preds[i] in all_data[i]["gold"] for i in range(len(preds)))
    output[model_name] = {
        "accuracy":    res["accuracy"],
        "correct":     int(correct),
        "total":       len(all_data),
        "predictions": preds,   # list of lowercase letter strings, e.g. ['a','b','c',…]
    }

with open("qwen14b_benchmark_results.json", "w") as f:
    json.dump(output, f, indent=2)
print("Saved: qwen14b_benchmark_results.json")

if "results_df" in vars():
    results_df.to_csv("qwen14b_benchmark_summary.csv", index=False)
    print("Saved: qwen14b_benchmark_summary.csv")

print("\nAnswer format : lowercase letter string ('a', 'b', 'c', …)")
print("Correct       : prediction is in the gold list (multi-answer aware)")
print("Random baseline: ~25–33 % depending on choice count per question")